# Outcome analysis

Frequency of the three task-round outcomes, by study arm, using `analysis/task_data.csv` (see `analysis/README.md` for how that file was built).

For each round, both partners independently choose a strategy: `C` (collaborative) or `I` (individual). The three possible outcomes are:

- **Successful collaboration** -- both partners choose `C`.
- **Mutual independence** -- both partners choose `I`.
- **Coordination failure** -- one partner chooses `C` and the other `I`.

In [1]:
import pandas as pd

task = pd.read_csv("task_data.csv")
task.head()

,arm,session,round,username_1,username_2,task_1,task_2,design_1,design_2,strategy_1,strategy_2,collabBelief_1,collabBelief_2,usedRobot_1,usedRobot_2,score_1,score_2
0,control,1,1,user0011,user0012,6,1,M,M,C,C,72,79,False,False,122.0,122.0
1,control,1,2,user0011,user0012,0,13,Y,L,I,C,83,60,False,False,50.0,-85.0
2,control,1,3,user0011,user0012,20,2,Y,L,I,C,50,70,False,False,50.0,-28.0
3,control,1,4,user0011,user0012,15,4,K,K,C,C,100,75,False,False,105.0,100.0
4,control,1,5,user0011,user0012,3,25,L,Y,C,I,0,60,False,False,11.0,50.0


## Drop missing data

A handful of rounds have `strategy_1`/`strategy_2` recorded as the literal string `"undefined"`, because that partner's submission failed to register that round (see `results/README.md#task_csv--decision-task-rounds`). Those rounds are dropped before computing outcome frequencies, since neither a collaborative/individual classification nor a real score is available for them.

In [2]:
missing = (task["strategy_1"] == "undefined") | (task["strategy_2"] == "undefined")
print(f"Dropping {missing.sum()} of {len(task)} rounds with an undefined strategy.")

task = task[~missing].copy()

Dropping 2 of 780 rounds with an undefined strategy.


## Classify each round's outcome

In [3]:
OUTCOME_ORDER = ["successful collaboration", "mutual independence", "coordination failure"]


def classify_outcome(row):
    s1, s2 = row["strategy_1"], row["strategy_2"]
    if s1 == "C" and s2 == "C":
        return "successful collaboration"
    if s1 == "I" and s2 == "I":
        return "mutual independence"
    return "coordination failure"


task["outcome"] = task.apply(classify_outcome, axis=1)
task["outcome"].value_counts().reindex(OUTCOME_ORDER)

outcome
successful collaboration    562
mutual independence         116
coordination failure        100
Name: count, dtype: int64

## Outcome frequency by arm

Counts and within-arm percentages, plus an `overall` row for both arms combined.

In [4]:
counts = task.groupby("arm")["outcome"].value_counts().unstack(fill_value=0)[OUTCOME_ORDER]
counts.loc["overall"] = counts.sum()

proportions = counts.div(counts.sum(axis=1), axis=0) * 100

outcome_summary = pd.DataFrame(index=counts.index)
outcome_summary["N"] = counts.sum(axis=1)
for outcome in OUTCOME_ORDER:
    outcome_summary[f"{outcome} (n)"] = counts[outcome]
    outcome_summary[f"{outcome} (%)"] = proportions[outcome].round(1)

outcome_summary

,N,successful collaboration (n),successful collaboration (%),mutual independence (n),mutual independence (%),coordination failure (n),coordination failure (%)
arm,,,,,,,
control,358,227,63.4,69,19.3,62,17.3
treatment,420,335,79.8,47,11.2,38,9.0
overall,778,562,72.2,116,14.9,100,12.9


## Inferential statistics: pair-level aggregation

Rounds within a pair aren't independent -- the same two partners produced all ~30 of their rounds together, so treating each round as an independent observation would overstate the sample size and understate uncertainty. `arm` is also assigned at the pair/session level, not the round level, which reinforces that the pair is the actual unit of randomization.

To respect that, each pair is first collapsed to a single summary: the fraction of *that pair's* rounds landing in each outcome. This gives one independent observation per pair (12 control, 14 treatment) rather than one per round (358 control, 420 treatment), at the cost of discarding round-to-round variation and any task-level covariates. A model that uses the round-level data directly (e.g. a mixed-effects model with a random intercept per pair) can recover that lost information as a follow-up analysis.

In [5]:
pair_summary = (
    task.groupby(["arm", "username_1", "username_2"])["outcome"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=OUTCOME_ORDER, fill_value=0)
)
pair_summary["n_rounds"] = task.groupby(["arm", "username_1", "username_2"]).size()
pair_summary = pair_summary.reset_index()

print(pair_summary["arm"].value_counts())
pair_summary

arm
treatment    14
control      12
Name: count, dtype: int64


outcome,arm,username_1,username_2,successful collaboration,mutual independence,coordination failure,n_rounds
0,control,user0011,user0012,0.433333,0.333333,0.233333,30
1,control,user0013,user0014,0.533333,0.066667,0.400000,30
2,control,user0023,user0024,0.966667,0.000000,0.033333,30
3,control,user0025,user0026,0.166667,0.333333,0.500000,30
4,control,user0027,user0028,0.172414,0.448276,0.379310,29
5,control,user0029,user0030,0.933333,0.000000,0.066667,30
6,control,user0035,user0036,1.000000,0.000000,0.000000,30
7,control,user0037,user0038,0.933333,0.000000,0.066667,30
8,control,user0043,user0044,0.766667,0.233333,0.000000,30
9,control,user0045,user0046,0.600000,0.300000,0.100000,30


### Two-sample tests

For each outcome, compare the 12 control-pair rates against the 14 treatment-pair rates with two complementary tests:

- **Welch's t-test** -- doesn't assume equal variance between arms, gives a mean difference and a p-value.
- **Mann-Whitney U** -- a nonparametric rank-sum test that doesn't assume normally-distributed rates, useful given how small and outcome-bounded (0-1) these samples are.

Effect size is reported as Cohen's *d* (mean difference divided by the pooled standard deviation). With only 12 and 14 pairs, treat p-values as indicative rather than definitive -- and note that testing all three outcomes here is not corrected for multiple comparisons.

In [6]:
from scipy import stats


def cohens_d(a, b):
    n_a, n_b = len(a), len(b)
    pooled_sd = (((n_a - 1) * a.var(ddof=1) + (n_b - 1) * b.var(ddof=1)) / (n_a + n_b - 2)) ** 0.5
    return (a.mean() - b.mean()) / pooled_sd


def compare_arms(outcome):
    control = pair_summary.loc[pair_summary["arm"] == "control", outcome]
    treatment = pair_summary.loc[pair_summary["arm"] == "treatment", outcome]
    t_stat, t_p = stats.ttest_ind(treatment, control, equal_var=False)
    u_stat, u_p = stats.mannwhitneyu(treatment, control, alternative="two-sided")
    return pd.Series({
        "control n": len(control),
        "control mean": control.mean(),
        "control sd": control.std(),
        "treatment n": len(treatment),
        "treatment mean": treatment.mean(),
        "treatment sd": treatment.std(),
        "diff (treatment - control)": treatment.mean() - control.mean(),
        "Cohen's d": cohens_d(treatment, control),
        "Welch t": t_stat,
        "Welch t p-value": t_p,
        "Mann-Whitney U": u_stat,
        "Mann-Whitney p-value": u_p,
    })


inferential_summary = pd.DataFrame({outcome: compare_arms(outcome) for outcome in OUTCOME_ORDER}).T
inferential_summary.round(4)

,control n,control mean,control sd,treatment n,treatment mean,treatment sd,diff (treatment - control),Cohen's d,Welch t,Welch t p-value,Mann-Whitney U,Mann-Whitney p-value
successful collaboration,12.0,0.6326,0.2925,14.0,0.7976,0.3045,0.1651,0.5519,1.4073,0.1723,119.5,0.0637
mutual independence,12.0,0.1940,0.1703,14.0,0.1119,0.1933,-0.0821,-0.4481,-1.1508,0.2612,59.0,0.1698
coordination failure,12.0,0.1735,0.1723,14.0,0.0905,0.1386,-0.0830,-0.5356,-1.3383,0.1951,50.0,0.0726


## Mixed-effects model: round-level data with a random intercept per pair

The pair-level test above is the most defensible test but throws away round-to-round variation (778 rounds collapsed to 26 numbers) and can't yet use round-level covariates. A mixed-effects model can use every round while still accounting for non-independence, by giving each pair its own random intercept -- rounds from the same pair share a baseline tendency, and `arm` is estimated on top of that.

**Basic model**: `successful_collaboration ~ arm`, with a random intercept for `pair`, no other covariates yet. `successful_collaboration` is `1` for a `C`/`C` round and `0` otherwise (this section only models the primary outcome; the other two can be added the same way later). Fit with `statsmodels`' `BinomialBayesMixedGLM`, a variational-Bayes logistic mixed model.

In [7]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

task["pair_id"] = task["username_1"] + "_" + task["username_2"]
task["successful_collaboration"] = (task["outcome"] == "successful collaboration").astype(int)

mixed_model = BinomialBayesMixedGLM.from_formula(
    "successful_collaboration ~ arm",
    vc_formulas={"pair": "0 + C(pair_id)"},
    data=task,
)
mixed_result = mixed_model.fit_vb()
print(mixed_result.summary())

                  Binomial Mixed GLM Results
                 Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------
Intercept           M     1.1757   0.1057                      
arm[T.treatment]    M     2.1100   0.1711                      
pair                V     0.9695   0.1349 2.637   2.013   3.453
Parameter types are mean structure (M) and variance structure
(V)
Variance parameters are modeled as log standard deviations


### A caveat that changes the conclusion

Notice how confident that looks: the `arm[T.treatment]` coefficient's posterior SD is tiny relative to the estimate, implying overwhelming significance -- far more confident than the pair-level test above, which only found a *trend* (Mann-Whitney p ≈ 0.06) for the same comparison.

That's a red flag rather than a genuine gain in power. `arm` is a **pair-level** covariate -- it's constant across all ~30 rounds within a pair, so the real information about its effect comes from only 26 independent pairs, no matter how many rounds are in the dataset. `BinomialBayesMixedGLM.fit_vb()` uses a *mean-field* variational approximation, which is well known to underestimate posterior uncertainty in hierarchical models -- and that failure mode is worst for exactly this kind of higher-level covariate, since mean-field VB assumes independence between the fixed effect and the random intercepts it's supposed to be disentangled from.

To check this suspicion, fit the same comparison with GEE (a population-averaged model with cluster-robust "sandwich" standard errors clustered on `pair_id`) -- a method that doesn't share this failure mode:

In [8]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

gee_model = smf.gee(
    "successful_collaboration ~ arm",
    groups="pair_id",
    data=task,
    family=sm.families.Binomial(),
)
gee_result = gee_model.fit()
print(gee_result.summary())

                                 GEE Regression Results                                
Dep. Variable:        successful_collaboration   No. Observations:                  778
Model:                                     GEE   No. clusters:                       26
Method:                            Generalized   Min. cluster size:                  29
                          Estimating Equations   Max. cluster size:                  30
Family:                               Binomial   Mean cluster size:                29.9
Dependence structure:             Independence   Num. iterations:                     2
Date:                         Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                        robust   Time:                         17:01:08
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.5498      0.348

### Which result to trust

GEE's `arm[T.treatment]` p-value lines up with the pair-level test (not significant at the conventional 0.05 threshold, though both point the same direction), confirming the mixed model's tiny standard error was an artifact of the variational fit, not a real result. **For the `arm` effect itself, the pair-level test and GEE are the trustworthy numbers here -- not the mixed model's reported significance.**

The mixed model isn't wasted effort, though: its random-intercept variance (`pair`, reported above) is a legitimate and useful estimate of how much pairs vary from each other in their baseline collaboration tendency, and once round-level covariates that actually *vary within a pair* are added (e.g. `task_difficulty`, `payoff_magnitude` from `task_summary.csv`), the mixed model becomes the right tool for estimating *those* effects with full round-level power -- that within-pair information doesn't suffer from the same failure mode, since those covariates aren't confounded with the random intercept the way a pair-level covariate is.

## Adding task difficulty

`analysis/task_summary.csv` assigns each real task index a `task_difficulty` tier (1-6, harsher downside payoffs at higher tiers). Each round involves two partners on two different tasks (`task_1`, `task_2`), so rather than two separate difficulty columns, this uses two derived measures:

- **`max_difficulty`** -- the harder of the two partners' tasks that round.
- **`diff_difficulty`** -- how mismatched the two partners' task difficulties are (absolute difference).

Both vary from round to round *within* a pair (unlike `arm`, which is constant within a pair), so they don't share `arm`'s confound with the random intercept -- the mixed model's estimates for these should be much more trustworthy than its estimate for `arm` was.

In [9]:
import numpy as np

task_summary = pd.read_csv("task_summary.csv")
difficulty_by_index = task_summary.set_index("task_index")["task_difficulty"]

# task_1/task_2 should only ever reference real tasks (0-29), which all
# have a numeric task_difficulty (not "n/a") in task_summary.csv
assert task["task_1"].isin(difficulty_by_index.index).all()
assert task["task_2"].isin(difficulty_by_index.index).all()

difficulty_1 = task["task_1"].map(difficulty_by_index).astype(int)
difficulty_2 = task["task_2"].map(difficulty_by_index).astype(int)
task["max_difficulty"] = np.maximum(difficulty_1, difficulty_2)
task["diff_difficulty"] = (difficulty_1 - difficulty_2).abs()

task[["max_difficulty", "diff_difficulty"]].describe()

,max_difficulty,diff_difficulty
count,778.000000,778.000000
mean,4.664524,2.336761
std,1.248650,1.247788
min,2.000000,1.000000
25%,4.000000,1.000000
50%,5.000000,2.000000
75%,6.000000,3.000000
max,6.000000,5.000000


### Success rate by arm and difficulty mismatch

A first look before modeling: does the relationship between `diff_difficulty` and success look different by arm?

In [10]:
(
    task.groupby(["arm", "diff_difficulty"])["successful_collaboration"]
    .mean()
    .unstack("arm")
    .round(3)
)

arm,control,treatment
diff_difficulty,,
1,0.703,0.807
2,0.698,0.777
3,0.611,0.810
4,0.479,0.821
5,0.417,0.750


### Model: does the treatment change the effect of difficulty mismatch?

The table above suggests it does -- control's success rate falls as the partners' tasks diverge in difficulty, while treatment's stays roughly flat. To test that formally: `successful_collaboration ~ arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c`.

An initial version of this model also included `arm:max_difficulty_c`, which was not significant (GEE p = 0.64) and is dropped here for a more parsimonious model.

`max_difficulty_c` and `diff_difficulty_c` are mean-centered before interacting, so `arm`'s main-effect coefficient is interpretable as the treatment effect at a *typical* task (average difficulty, average mismatch) rather than extrapolating to `diff_difficulty = 0`, which never actually occurs in this data (`diff_difficulty` ranges 1-5).

In [11]:
task["max_difficulty_c"] = task["max_difficulty"] - task["max_difficulty"].mean()
task["diff_difficulty_c"] = task["diff_difficulty"] - task["diff_difficulty"].mean()

difficulty_formula = "successful_collaboration ~ arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c"

gee_difficulty = smf.gee(
    difficulty_formula, groups="pair_id", data=task, family=sm.families.Binomial(),
).fit()
print(gee_difficulty.summary())

                                 GEE Regression Results                                
Dep. Variable:        successful_collaboration   No. Observations:                  778
Model:                                     GEE   No. clusters:                       26
Method:                            Generalized   Min. cluster size:                  29
                          Estimating Equations   Max. cluster size:                  30
Family:                               Binomial   Mean cluster size:                29.9
Dependence structure:             Independence   Num. iterations:                     2
Date:                         Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                        robust   Time:                         17:01:08
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
In

In [12]:
mixed_difficulty = BinomialBayesMixedGLM.from_formula(
    difficulty_formula,
    vc_formulas={"pair": "0 + C(pair_id)"},
    data=task,
).fit_vb()
print(mixed_difficulty.summary())

                           Binomial Mixed GLM Results
                                   Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------------------------
Intercept                             M     1.3670   0.1155                      
arm[T.treatment]                      M     2.2366   0.1841                      
max_difficulty_c                      M    -0.8056   0.0984                      
diff_difficulty_c                     M    -0.1513   0.0926                      
arm[T.treatment]:diff_difficulty_c    M     0.5228   0.1471                      
pair                                  V     1.0881   0.1346 2.969   2.268   3.886
Parameter types are mean structure (M) and variance structure (V)
Variance parameters are modeled as log standard deviations


### Interpretation

GEE (the trustworthy standard errors here) and the mixed model agree in sign and relative magnitude throughout, which is expected -- `max_difficulty` and `diff_difficulty` vary *within* pairs, so they don't have `arm`'s pair-level confound with the random intercept.

- **`max_difficulty_c` is a robust, strongly significant predictor** (GEE p < 0.001): harder tasks reduce successful collaboration, consistent with their harsher downside payoffs.
- **`diff_difficulty_c` (the within-control-arm slope) is negative but not significant on its own** -- consistent with the descriptive drop above, but only 12 control pairs carry that estimate.
- **`arm:diff_difficulty_c` is significant** (GEE p ≈ 0.02): the treatment arm's slope for difficulty mismatch is reliably less negative (flatter) than control's. Concretely, control's success rate declines noticeably as the partners' tasks diverge in difficulty, while treatment's stays roughly flat -- the treatment appears to specifically buffer against the harm of difficulty asymmetry between partners, rather than just uniformly raising the collaboration rate.
- **`arm`'s main effect (at a typical task) remains non-significant**, same conclusion as the basic model: there's no reliable *uniform* treatment effect, but there is a reliable *difference in how the two arms respond to difficulty mismatch*.

## Robustness check: excluding pairs that never used the robot

Two of the 14 treatment pairs never touched the robot recommendation at all (see `robot_use_analysis.ipynb`). Since the robot is the mechanism most likely responsible for any treatment effect, it's worth checking whether these two "non-adopter" pairs are diluting the results above.

**This is a post-hoc, per-protocol-style check, not a causal per-protocol estimate** -- whether a pair chose to use the robot wasn't randomly assigned, so it could correlate with other traits of the pair (engagement, trust, personality) that independently affect the outcome. A rigorous version of "the effect among compliers" would need something like an instrumental-variable / Complier Average Causal Effect (CACE) approach. Treat this as a robustness/sensitivity check on the results above, not a replacement for them.

In [13]:
never_used_robot_pairs = {"user0031_user0032", "user0039_user0040"}
task_excl = task[~task["pair_id"].isin(never_used_robot_pairs)].copy()

print(f"Full sample: {task['pair_id'].nunique()} pairs ({len(task)} rounds)")
print(f"Excluding non-adopters: {task_excl['pair_id'].nunique()} pairs ({len(task_excl)} rounds)")

# pair-level test, restricted to successful collaboration (the outcome
# already tested above)
pair_excl = (
    task_excl.groupby(["arm", "pair_id"])["outcome"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)
control_vals = pair_excl.xs("control", level="arm")["successful collaboration"]
treatment_vals = pair_excl.xs("treatment", level="arm")["successful collaboration"]

t_stat, t_p = stats.ttest_ind(treatment_vals, control_vals, equal_var=False)
u_stat, u_p = stats.mannwhitneyu(treatment_vals, control_vals, alternative="two-sided")
print(f"\nPair-level (successful collaboration): control mean={control_vals.mean():.3f} (n={len(control_vals)}), "
      f"treatment mean={treatment_vals.mean():.3f} (n={len(treatment_vals)})")
print(f"Welch t={t_stat:.3f} p={t_p:.4f} | Mann-Whitney U={u_stat:.1f} p={u_p:.4f}")

Full sample: 26 pairs (778 rounds)
Excluding non-adopters: 24 pairs (718 rounds)

Pair-level (successful collaboration): control mean=0.633 (n=12), treatment mean=0.819 (n=12)
Welch t=1.564 p=0.1321 | Mann-Whitney U=106.0 p=0.0469


In [14]:
gee_basic_excl = smf.gee(
    "successful_collaboration ~ arm", groups="pair_id", data=task_excl, family=sm.families.Binomial(),
).fit()
print("=== GEE (basic model), excluding non-adopters ===")
print(gee_basic_excl.summary().tables[1])

gee_difficulty_excl = smf.gee(
    difficulty_formula, groups="pair_id", data=task_excl, family=sm.families.Binomial(),
).fit()
print("\n=== GEE (difficulty-interaction model), excluding non-adopters ===")
print(gee_difficulty_excl.summary().tables[1])

=== GEE (basic model), excluding non-adopters ===
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.5498      0.348      1.581      0.114      -0.132       1.231
arm[T.treatment]     0.9628      0.648      1.485      0.137      -0.308       2.233

=== GEE (difficulty-interaction model), excluding non-adopters ===
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              0.6043      0.377      1.604      0.109      -0.134       1.342
arm[T.treatment]                       0.9756      0.681      1.433      0.152      -0.358       2.310
max_difficulty_c                      -0.4200      0.083     -5.047      0.000      -0.583      -0.257
diff_difficulty_c          

### Why the interaction changes so much

The `arm:diff_difficulty_c` interaction gets noticeably stronger once the two non-adopter pairs are dropped. Looking at those two pairs individually explains why: they pull in opposite, uninformative directions rather than both diluting the effect the same way.

In [15]:
for pair in sorted(never_used_robot_pairs):
    u1, u2 = pair.split("_")
    sub = task[(task["username_1"] == u1) & (task["username_2"] == u2)]
    print(f"--- {pair} (never used the robot) ---")
    print(sub.groupby("diff_difficulty")["successful_collaboration"].mean().to_string())
    print()

--- user0031_user0032 (never used the robot) ---
diff_difficulty
1    0.500000
2    0.500000
3    0.166667
4    0.000000
5    0.000000

--- user0039_user0040 (never used the robot) ---
diff_difficulty
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0



`user0031`/`user0032` shows a clean, steep *control-like* decline as `diff_difficulty` rises (0.50 -> 0.50 -> 0.17 -> 0 -> 0) despite being assigned to treatment -- consistent with getting none of the buffering effect, since they never engaged with the tool that plausibly provides it. `user0039`/`user0040` is flat at a perfect 1.0 across every `diff_difficulty` value -- not evidence against the effect, just uninformative (no variation to estimate a slope from). Together, one pair was actively working against the interaction and the other contributed nothing either way, so removing both sharpens the estimate without particularly changing the point estimate's plausibility.

### Summary

| | Full sample (14 treatment pairs) | Excluding 2 non-adopters (12 pairs) |
|---|---|---|
| Pair-level Mann-Whitney (success) | p = 0.064 | p = 0.047 |
| GEE `arm` (basic model) | coef 0.82, p = 0.169 | coef 0.96, p = 0.137 |
| GEE `max_difficulty_c` | coef -0.43, p < 0.001 | coef -0.42, p < 0.001 |
| GEE `arm:diff_difficulty_c` | coef 0.29, p = 0.021 | coef 0.37, p = 0.0002 |

The basic `arm` effect barely moves (one dropped pair was low-performing, the other a perfect performer -- they roughly cancel in the mean). The **interaction effect gets substantially stronger**, consistent with a pair that never adopted the tool behaving like a control pair despite treatment assignment. As stated above, this is a sensitivity check on a non-randomly-selected subgroup, not a causal claim about the effect "among users who would comply" -- reported here as a robustness signal alongside the main results, not a replacement for them.

## Coordination failure

Everything above modeled `successful_collaboration` at the round level. The pair-level test at the very top of this notebook already covered `coordination failure` too (control mean 17.3%, treatment mean 9.0%, Cohen's *d* = -0.54, Mann-Whitney p = 0.073 -- a trend, not significant), but round-level modeling with covariates was only built out for the primary outcome. This extends the same approach -- GEE trusted over the mixed model's standard error for `arm`, both trusted for within-pair-varying covariates -- to `coordination_failure` specifically.

Coordination failure is arguably the outcome of most practical interest on its own: it's the case where one partner is left exposed to the downside (`V_A^{CI}`) while the other captures the safe payoff regardless (`V_Y`) -- the same asymmetry that motivated `belief_manipulation_analysis.ipynb`. A treatment effect on `successful_collaboration` could in principle come from reducing coordination failure, reducing mutual independence, or both -- this section isolates the coordination-failure channel specifically. (The complementary mutual-independence channel isn't modeled here, since it wasn't requested, but the same approach would apply.)

In [16]:
task["coordination_failure"] = (task["outcome"] == "coordination failure").astype(int)

gee_cf_basic = smf.gee(
    "coordination_failure ~ arm", groups="pair_id", data=task, family=sm.families.Binomial(),
).fit()
print("=== GEE ===")
print(gee_cf_basic.summary().tables[1])

mixed_cf_basic = BinomialBayesMixedGLM.from_formula(
    "coordination_failure ~ arm", vc_formulas={"pair": "0 + C(pair_id)"}, data=task,
).fit_vb()
print("\n=== Mixed model (VB) ===")
print(mixed_cf_basic.summary())

=== GEE ===
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -1.5632      0.333     -4.700      0.000      -2.215      -0.911
arm[T.treatment]    -0.7446      0.547     -1.362      0.173      -1.816       0.327



=== Mixed model (VB) ===
                  Binomial Mixed GLM Results
                 Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------
Intercept           M    -2.1207   0.1189                      
arm[T.treatment]    M    -1.4135   0.1870                      
pair                V     0.5813   0.1359 1.788   1.363   2.347
Parameter types are mean structure (M) and variance structure
(V)
Variance parameters are modeled as log standard deviations


Same anti-conservatism as before: the mixed model's `arm` standard error (0.187) is a third the size of GEE's (0.547), implying spurious near-certainty. GEE says `arm` alone is not a significant predictor of coordination failure (p = 0.173) -- consistent with the pair-level test above.

### Coordination failure rate by arm and difficulty mismatch

In [17]:
(
    task.groupby(["arm", "diff_difficulty"])["coordination_failure"]
    .mean()
    .unstack("arm")
    .round(3)
)

arm,control,treatment
diff_difficulty,,
1,0.119,0.071
2,0.167,0.125
3,0.167,0.083
4,0.250,0.071
5,0.333,0.107


Control's coordination-failure rate climbs steadily as `diff_difficulty` rises (11.9% -> 33.3%), the mirror image of its declining success rate above. Treatment's stays low and roughly flat (7-13%) across the same range -- the same buffering shape found for `successful_collaboration`, now visible in this specific failure mode.

### Model: does the treatment change the effect of difficulty mismatch on coordination failure?

Same formula as the `successful_collaboration` model: `coordination_failure ~ arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c`. An `arm:max_difficulty_c` term was tested and dropped -- not close to significant (GEE p = 0.96), same as for the primary outcome.

In [18]:
cf_difficulty_formula = "coordination_failure ~ arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c"

gee_cf_difficulty = smf.gee(
    cf_difficulty_formula, groups="pair_id", data=task, family=sm.families.Binomial(),
).fit()
print("=== GEE ===")
print(gee_cf_difficulty.summary().tables[1])

mixed_cf_difficulty = BinomialBayesMixedGLM.from_formula(
    cf_difficulty_formula, vc_formulas={"pair": "0 + C(pair_id)"}, data=task,
).fit_vb()
print("\n=== Mixed model (VB) ===")
print(mixed_cf_difficulty.summary())

=== GEE ===
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             -1.6100      0.344     -4.687      0.000      -2.283      -0.937
arm[T.treatment]                      -0.6982      0.554     -1.261      0.207      -1.784       0.387
max_difficulty_c                      -0.0184      0.129     -0.142      0.887      -0.272       0.235
diff_difficulty_c                      0.3081      0.069      4.459      0.000       0.173       0.443
arm[T.treatment]:diff_difficulty_c    -0.2742      0.164     -1.671      0.095      -0.596       0.047



=== Mixed model (VB) ===
                           Binomial Mixed GLM Results
                                   Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------------------------
Intercept                             M    -2.1958   0.1207                      
arm[T.treatment]                      M    -1.3895   0.1881                      
max_difficulty_c                      M    -0.0174   0.0984                      
diff_difficulty_c                     M     0.3825   0.0941                      
arm[T.treatment]:diff_difficulty_c    M    -0.3460   0.1503                      
pair                                  V     0.6015   0.1358 1.825   1.391   2.395
Parameter types are mean structure (M) and variance structure (V)
Variance parameters are modeled as log standard deviations


### Interpretation

- **`max_difficulty_c` is not a significant predictor of coordination failure** (GEE coef -0.018, p = 0.887) -- a contrast with `successful_collaboration`, where it was strongly significant. Harder tasks reduce success, but that mostly seems to route through *mutual* withdrawal into individual play rather than *asymmetric* breakdown between partners (consistent with the descriptive rise in the raw outcome counts for `mutual independence` reported at the top of this notebook).
- **`diff_difficulty_c` (the control-arm slope) is significant and positive** (GEE coef 0.308, p < 0.001): as the partners' task difficulties diverge, coordination failure becomes more likely in control -- the direct complement of the declining success rate found earlier, and intuitive on its own: when one partner faces a much harder task than the other, they're more likely to reach different conclusions about whether collaborating is worth the risk.
- **`arm:diff_difficulty_c` is a trend, not significant in the full sample** (GEE coef -0.274, p = 0.095), but consistent in sign and comparable in magnitude to every other version of the buffering interaction found in this analysis (`successful_collaboration` above, and independently in `efficiency_analysis.ipynb` and `belief_analysis.ipynb`): treatment's slope for difficulty mismatch is flatter than control's, i.e. less prone to escalating into coordination failure as mismatch grows.
- **`arm`'s main effect remains non-significant**, same conclusion as everywhere else: no reliable *uniform* reduction in coordination failure, but a real difference in how each arm's coordination-failure rate responds to difficulty mismatch.

### Robustness check: excluding non-adopter pairs

Same two pairs, same rationale as the `successful_collaboration` robustness check above (`task_excl`, already computed).

In [19]:
task_excl["coordination_failure"] = (task_excl["outcome"] == "coordination failure").astype(int)

print("=== GEE (basic model), excluding non-adopters ===")
print(smf.gee("coordination_failure ~ arm", groups="pair_id", data=task_excl, family=sm.families.Binomial())
      .fit().summary().tables[1])

print("\n=== GEE (difficulty-interaction model), excluding non-adopters ===")
print(smf.gee(cf_difficulty_formula, groups="pair_id", data=task_excl, family=sm.families.Binomial())
      .fit().summary().tables[1])

control_vals = pair_excl.xs("control", level="arm")["coordination failure"]
treatment_vals = pair_excl.xs("treatment", level="arm")["coordination failure"]
t_stat, t_p = stats.ttest_ind(treatment_vals, control_vals, equal_var=False)
u_stat, u_p = stats.mannwhitneyu(treatment_vals, control_vals, alternative="two-sided")
print(f"\nPair-level (coordination failure), excluding non-adopters: control mean={control_vals.mean():.3f} (n={len(control_vals)}), "
      f"treatment mean={treatment_vals.mean():.3f} (n={len(treatment_vals)})")
print(f"Welch t={t_stat:.3f} p={t_p:.4f} | Mann-Whitney U={u_stat:.1f} p={u_p:.4f}")

=== GEE (basic model), excluding non-adopters ===
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -1.5632      0.333     -4.700      0.000      -2.215      -0.911
arm[T.treatment]    -0.7988      0.599     -1.334      0.182      -1.972       0.375

=== GEE (difficulty-interaction model), excluding non-adopters ===
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                             -1.6098      0.344     -4.684      0.000      -2.283      -0.936
arm[T.treatment]                      -0.7591      0.606     -1.253      0.210      -1.947       0.428
max_difficulty_c                      -0.0017      0.136     -0.013      0.990      -0.268       0.264
diff_difficulty_c          

The `arm:diff_difficulty_c` interaction, only a trend in the full sample (p = 0.095), becomes clearly significant once the two non-adopter pairs are excluded (p = 0.007) -- the same pattern, and the same explanation, as the `successful_collaboration` robustness check above: `user0031_user0032` behaves like a control pair (its coordination-failure rate climbing with difficulty mismatch despite treatment assignment) since it never engaged with the tool that plausibly drives the buffering effect, while `user0039_user0040` is uninformative (0% coordination failure at every difficulty level). Removing both sharpens the same estimate rather than changing its substance -- this is now the outcome-specific confirmation of a pattern already found four other ways in this analysis.

## Joint multinomial model: all three outcomes at once

Everything above modeled the outcomes one binary comparison at a time (`successful_collaboration` vs. everything else, then `coordination_failure` vs. everything else). Since the three outcomes are mutually exclusive categories of one underlying choice, they can instead be compared **simultaneously** with a multinomial logit -- one model, one set of covariates, all three categories at once, with one category held out as the reference.

`statsmodels`' `NominalGEE` extends this to the clustered setting used throughout this notebook: a population-averaged multinomial model with cluster-robust ("sandwich") standard errors on `pair_id`, the same GEE machinery already trusted here for the `arm` effect. `successful_collaboration` is set as the reference category, so every coefficient is interpreted as "this outcome vs. successful collaboration."

**On mixed effects**: a true mixed-effects (random-intercept) multinomial model isn't available in `statsmodels`, and would require PyMC/Stan or R (`brms`, `mclogit`) to fit properly. Given that `BinomialBayesMixedGLM`'s variational fit already turned out to badly underestimate the standard error for `arm` (a pair-level covariate) earlier in this notebook, a from-scratch mixed multinomial model wouldn't obviously be more trustworthy here even if it were available -- GEE's marginal, cluster-robust approach is the one already validated as reliable for this exact clustering structure, so it's used here rather than chasing random effects for their own sake.

In [20]:
from statsmodels.genmod.generalized_estimating_equations import NominalGEE

outcome_code_map = {"mutual independence": 0, "coordination failure": 1, "successful collaboration": 2}
task["outcome_code"] = task["outcome"].map(outcome_code_map)

nominal_basic = NominalGEE.from_formula("outcome_code ~ arm", groups="pair_id", data=task).fit()
print(nominal_basic.summary())

                           NominalGEE Regression Results                           
Dep. Variable:                           y   No. Observations:                 1556
Model:                          NominalGEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                       _Multinomial   Mean cluster size:                59.8
Dependence structure:  NominalIndependence   Num. iterations:                    12
Date:                     Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         17:01:10
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept[0.0]           -1.1908      0.301     -3.956      0.00

`[0.0]` coefficients are mutual independence vs. successful collaboration; `[1.0]` are coordination failure vs. successful collaboration. Neither `arm` contrast reaches significance here (p = 0.14, p = 0.11) -- consistent with the separate binomial GEEs above, as a sanity check that the joint model reduces to the same conclusions when there's no covariate to gain efficiency from yet.

### Adding task difficulty

Same covariates as above. An `arm:max_difficulty_c` term was tested for both contrasts and dropped -- not close to significant either way (GEE p = 0.80 and p = 0.78).

In [21]:
nominal_formula = "outcome_code ~ arm + max_difficulty_c + diff_difficulty_c + arm:diff_difficulty_c"
nominal_difficulty = NominalGEE.from_formula(nominal_formula, groups="pair_id", data=task).fit()
print(nominal_difficulty.summary())

                           NominalGEE Regression Results                           
Dep. Variable:                           y   No. Observations:                 1556
Model:                          NominalGEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                       _Multinomial   Mean cluster size:                59.8
Dependence structure:  NominalIndependence   Num. iterations:                    18
Date:                     Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         17:01:10
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept[0.0]              

### Interpretation

Fitting all three outcomes jointly separates two mechanisms that the earlier one-outcome-at-a-time models could only imply separately:

- **`max_difficulty_c` is a significant predictor of mutual independence vs. success** (coef 0.741, p < 0.001) **but not of coordination failure vs. success** (coef 0.092, p = 0.483). Harder tasks specifically push pairs toward *both* choosing individually, not toward one partner exploiting or being exploited by the other.
- **`diff_difficulty_c` shows the opposite split**: not significant for mutual independence vs. success (coef -0.024, p = 0.846), but significant for coordination failure vs. success (coef 0.322, p < 0.001). Difficulty *mismatch* between partners specifically drives coordination failure, not mutual withdrawal -- makes sense, since a mismatch is exactly the situation where one partner's task looks safe to collaborate on while the other's doesn't.
- **`arm:diff_difficulty_c` is significant for coordination failure vs. success** (coef -0.345, p = 0.035) but not for mutual independence vs. success (coef -0.232, p = 0.125). This is the same buffering interaction found throughout this analysis -- but here it reaches significance **in the full 14-pair treatment sample**, without needing the non-adopter exclusion that the separate binomial GEE required (p = 0.095 there). Modeling all three categories jointly is more statistically efficient than binarizing twice, and the sharper estimate here is a direct benefit of that.
- **`arm`'s main effect remains non-significant for both contrasts**, the same conclusion as every model in this notebook: no reliable *uniform* treatment effect, only a reliable *difference in how each arm responds to difficulty*.

Taken together: task difficulty and difficulty mismatch push pairs away from successful collaboration through two distinct, separable channels -- overall difficulty toward mutual independence, mismatch toward coordination failure -- and the treatment's buffering effect against mismatch is specifically a buffering effect against *coordination failure*, not against mutual independence.

### Sensitivity check: excluding task index 19

`data/README.md` documents a payoff anomaly in task index 19: its highest-upside design (`Design M`, ranked `A`) has a downside payoff of -68, identical to task index 14's, instead of a harsher value consistent with its own (higher) difficulty tier -- most likely a copy-paste error in the original task table, not a data-collection issue. Task 19 is always paired with task 5 in this dataset, so every round involving it sits at exactly `diff_difficulty = 2`, `max_difficulty = 4` -- a fixed point, not spread across the range. Since the `arm:diff_difficulty_c` interaction above is exactly the kind of result this anomaly could distort, it's worth checking whether it holds up without those rounds.

In [22]:
involves_task_19 = (task["task_1"] == 19) | (task["task_2"] == 19)
task_excl19 = task[~involves_task_19].copy()

print(f"Full sample: {task['pair_id'].nunique()} pairs ({len(task)} rounds)")
print(f"Excluding task 19: {task_excl19['pair_id'].nunique()} pairs "
      f"({len(task_excl19)} rounds, {involves_task_19.sum()} dropped)")

nominal_excl19 = NominalGEE.from_formula(nominal_formula, groups="pair_id", data=task_excl19).fit()
print(nominal_excl19.summary())

Full sample: 26 pairs (778 rounds)
Excluding task 19: 26 pairs (726 rounds, 52 dropped)


                           NominalGEE Regression Results                           
Dep. Variable:                           y   No. Observations:                 1452
Model:                          NominalGEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  54
                      Estimating Equations   Max. cluster size:                  56
Family:                       _Multinomial   Mean cluster size:                55.8
Dependence structure:  NominalIndependence   Num. iterations:                    18
Date:                     Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         17:01:11
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept[0.0]              

### Interpretation

The mutual-independence/coordination-failure split holds up essentially unchanged: `max_difficulty_c` still strongly predicts mutual independence vs. success (coef 0.720, p < 0.001) and not coordination failure (coef 0.107, p = 0.418); `diff_difficulty_c` still strongly predicts coordination failure and not mutual independence (coef 0.312, p < 0.001). `arm`'s main effect remains non-significant for both, as everywhere else.

**`arm:diff_difficulty_c` for coordination failure vs. success drops from significant to a trend**: coef -0.345 (p = 0.035) in the full sample vs. coef -0.324 (p = 0.070) excluding task 19. The coefficient itself barely moves (~6% smaller), but the standard error grows more than the 6.7%-smaller sample alone would explain (0.164 -> 0.179), so this specific result is more sensitive to the anomalous task than the sample-size loss suggests -- worth being aware of before leaning on that particular p-value. The direction and rough magnitude of the buffering effect are unchanged, and every other conclusion in this section is robust to the exclusion; it's specifically the significance threshold on this one contrast that the anomaly is close enough to affect.

### Sensitivity check: recoding task index 19 to its actual difficulty tier

Excluding task 19 discards real information along with the error. A sharper alternative: task 19's `u_A` (0.7024) matches the tier-3 block (tasks 10-14, `u_A` 0.7005-0.7068) almost exactly, not tier 4's ~0.75 -- and participant behavior on task 19 confirms this (see prior discussion: its `chose_C` rate is statistically indistinguishable from the tier-3 block). So rather than dropping task 19's rounds, this refits the joint model with task 19 *recoded* to `task_difficulty = 3`, correcting `max_difficulty`/`diff_difficulty` for every round involving it to reflect its true risk profile.

In [23]:
difficulty_recoded = difficulty_by_index.to_dict()
difficulty_recoded[19] = 3

d1_recoded = task["task_1"].map(difficulty_recoded).astype(int)
d2_recoded = task["task_2"].map(difficulty_recoded).astype(int)

task_recoded = task.copy()
task_recoded["max_difficulty"] = np.maximum(d1_recoded, d2_recoded)
task_recoded["diff_difficulty"] = (d1_recoded - d2_recoded).abs()
task_recoded["max_difficulty_c"] = task_recoded["max_difficulty"] - task_recoded["max_difficulty"].mean()
task_recoded["diff_difficulty_c"] = task_recoded["diff_difficulty"] - task_recoded["diff_difficulty"].mean()

nominal_recoded = NominalGEE.from_formula(nominal_formula, groups="pair_id", data=task_recoded).fit()
print(nominal_recoded.summary())

                           NominalGEE Regression Results                           
Dep. Variable:                           y   No. Observations:                 1556
Model:                          NominalGEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                       _Multinomial   Mean cluster size:                59.8
Dependence structure:  NominalIndependence   Num. iterations:                    18
Date:                     Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         17:01:11
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept[0.0]              

### Interpretation

Recoding strengthens the buffering interaction rather than weakening it -- the opposite direction from simply excluding task 19:

| `arm:diff_difficulty_c` | Full sample (task 19 = tier 4) | Excluding task 19 | **Recoding task 19 = tier 3** |
|---|---|---|---|
| Mutual independence vs. success | coef -0.232, p = 0.125 | coef -0.218, p = 0.160 | coef -0.248, p = 0.071 |
| Coordination failure vs. success | coef -0.345, p = 0.035 | coef -0.324, p = 0.070 | **coef -0.372, p = 0.012** |

Every other coefficient (`max_difficulty_c`, `diff_difficulty_c` main effects, `arm` main effects) is essentially unchanged from the original model in both magnitude and significance.

This makes sense given what excluding task 19 could not distinguish: task 19's rounds aren't noise to be discarded, they're *mismeasured* -- tagged with a `diff_difficulty` that doesn't reflect the actual risk mismatch participants faced. Dropping them removes real information along with the error, which is why exclusion made the estimate noisier (larger SE) without changing the point estimate much. Correcting the label instead fixes the measurement itself, letting the true relationship come through more clearly: the coordination-failure buffering interaction, which needed either the full multinomial model or a non-adopter exclusion to reach significance in the original (mislabeled) data, is significant here on its own, in the full 14-pair sample, with no exclusions at all.